# ETL Pipeline: OpenBind EV-A71 2A Dataset to Queryable Database


datasource
- https://zenodo.org/records/20026661
- used only EV-A71_2A_metadata.csv, for pipeline: explore, data cleaning, load to db, and query

### Phase 1: download OpenBind EV-A71 2A dataset

In [ ]:
# Install zenodo_get to download records directly from Zenodo
!pip install -q zenodo-get

# Download the OpenBind dataset from Zenodo record 20026661
!zenodo_get 20026661

import pandas as pd
import os
import zipfile

print("Extracting downloaded ZIP archive...")
zip_file = 'OpenBind_EV-A71_2A.zip'
if os.path.exists(zip_file):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall('openbind_data')

# --- 2. Data Extraction ---
print("\nFound CSV files in extracted data:")
csv_files = []
for root, dirs, files in os.walk('openbind_data'):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))

for f in csv_files:
    print(f" - {f}")

In [4]:


print("\nExtracting data into DataFrame...")
if len(csv_files) > 0:
    metadata_file = csv_files[0]
    df_metadata = pd.read_csv(metadata_file)
    print(f"\nLoaded Metadata from: {metadata_file}")
    display(df_metadata.head(3))
else:
    print("\nCould not find any CSV files to load.")


Extracting data into DataFrame...

Loaded Metadata from: openbind_data/OpenBind_EV-A71_2A/EV-A71_2A_metadata.csv


,complex_name,compound_group,smiles,experimental_pKD,covalent,fragment_screen,pb_valid_prepared,pb_valid_ref,suspected_artefact,sucos_shape,pocket_qcov,sucos_shape_pocket_qcov
0,A71EV2A-x0152a,x0152a_x0152b,O=C1CN(C(=O)COc2ccccc2)CCN1,NaN,False,True,True,True,True,40.5843,71.4286,28.9888
1,A71EV2A-x0152b,x0152a_x0152b,O=C1CN(C(=O)COc2ccccc2)CCN1,NaN,False,True,False,False,False,45.6884,38.4615,17.5725
2,A71EV2A-x0188a,x0188a,CC[C@@H]1CN(S(=O)(=O)N(C)C)CCO1,NaN,False,True,True,True,False,57.7371,22.2222,12.8305


### Data Exploration
display cols
- each row is a complex built from target i.e. EV-A71 2A protein with a drug candidate
- each col is a measurement for the complex, some are manual measurements (e.g. pKD, fragment_screen), others are computationally generated data based on manual data.

In [5]:
# Explore the metadata dataset
print("--- Metadata Dataset Info ---")
df_metadata.info()
print("\n--- Metadata Dataset Summary ---")
display(df_metadata.describe(include='all'))

# Check for missing values
print("\n--- Missing Values in Metadata ---")
display(df_metadata.isnull().sum())

--- Metadata Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   complex_name             925 non-null    object 
 1   compound_group           925 non-null    object 
 2   smiles                   925 non-null    object 
 3   experimental_pKD         649 non-null    float64
 4   covalent                 925 non-null    bool   
 5   fragment_screen          925 non-null    bool   
 6   pb_valid_prepared        925 non-null    bool   
 7   pb_valid_ref             925 non-null    bool   
 8   suspected_artefact       925 non-null    bool   
 9   sucos_shape              925 non-null    float64
 10  pocket_qcov              925 non-null    float64
 11  sucos_shape_pocket_qcov  925 non-null    float64
dtypes: bool(5), float64(4), object(3)
memory usage: 55.2+ KB

--- Metadata Dataset Summary ---


,complex_name,compound_group,smiles,experimental_pKD,covalent,fragment_screen,pb_valid_prepared,pb_valid_ref,suspected_artefact,sucos_shape,pocket_qcov,sucos_shape_pocket_qcov
count,925,925,925,649.000000,925,925,925,925,925,925.000000,925.000000,925.000000
unique,925,699,699,NaN,2,2,2,2,2,NaN,NaN,NaN
top,A71EV2A-x0152a,x4829a_x4829b_x4903a_x4903b,COc1ccc(Cl)cc1OC(=O)N1CCC[C@H](C(N)=O)C1,NaN,False,False,True,True,False,NaN,NaN,NaN
freq,1,4,4,NaN,923,812,919,897,886,NaN,NaN,NaN
mean,NaN,NaN,NaN,5.090018,NaN,NaN,NaN,NaN,NaN,43.597690,86.362811,37.134471
std,NaN,NaN,NaN,0.835982,NaN,NaN,NaN,NaN,NaN,5.964828,17.962106,8.183696
min,NaN,NaN,NaN,3.436900,NaN,NaN,NaN,NaN,NaN,29.669100,0.000000,0.000000
25%,NaN,NaN,NaN,4.556500,NaN,NaN,NaN,NaN,NaN,40.023400,88.888900,34.943000
50%,NaN,NaN,NaN,4.939700,NaN,NaN,NaN,NaN,NaN,43.186600,92.000000,38.624800
75%,NaN,NaN,NaN,5.601900,NaN,NaN,NaN,NaN,NaN,46.333300,92.592600,41.791900



--- Missing Values in Metadata ---


,0
complex_name,0
compound_group,0
smiles,0
experimental_pKD,276
covalent,0
fragment_screen,0
pb_valid_prepared,0
pb_valid_ref,0
suspected_artefact,0
sucos_shape,0


### Phase 2: Transformation
- filter out rows w/ missing pKD
- deduplicate: same smiles i.e. 2d structure, keeps only row w/ highest pKD i.e. binding affinity

In [7]:
# Filter out incomplete rows where experimental binding affinity (pKD) is missing
df_clean = df_metadata.dropna(subset=['experimental_pKD']).copy()

# Optional: Keep only the highest affinity entry if there are duplicate SMILES strings
df_clean = df_clean.sort_values('experimental_pKD', ascending=False).drop_duplicates(subset=['smiles'])

print("Transformed Dataset (Filtered missing pKD & deduplicated by SMILES):")
print(f"Original row count: {len(df_metadata)}")
print(f"Cleaned row count: {len(df_clean)}")
display(df_clean.head())

Transformed Dataset (Filtered missing pKD & deduplicated by SMILES):
Original row count: 925
Cleaned row count: 499


,complex_name,compound_group,smiles,experimental_pKD,covalent,fragment_screen,pb_valid_prepared,pb_valid_ref,suspected_artefact,sucos_shape,pocket_qcov,sucos_shape_pocket_qcov
848,A71EV2A-x7259a,x7259a,CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](...,7.9380,False,False,True,True,False,37.4535,92.3077,34.5725
826,A71EV2A-x7135a,x7135a,CSc1ncnc(N2CCC[C@@H]2C(C)C)c1CC(=O)N1CCC[C@H](...,7.7602,False,False,True,True,False,37.3312,92.3077,34.4596
772,A71EV2A-x6832a,x6832a,CSc1ncnc(N2CCC[C@@H]2C(N)=O)c1CC(=O)N1CCC[C@H]...,7.7144,False,False,True,True,False,40.5085,92.3077,37.3924
825,A71EV2A-x7132a,x7132a,CSc1ncnc(N2CCC[C@@H]2Cc2ccnn2C)c1CC(=O)N1CCC[C...,7.3594,False,False,True,True,False,38.1587,92.3077,35.2235
800,A71EV2A-x7011a,x7011a,CC[C@H]1CCCN1c1ncnc(SC)c1CC(=O)N1CCC[C@H](C(N)...,7.3026,False,False,True,True,False,36.6061,92.3077,33.7902


### Phase 3: Loading
- load the cleaned data into a SQLite db in colab, run SQL query to get top5 most promising complex i.e. highest affinities

In [ ]:
import sqlite3
import pandas as pd

# Initialize a lightweight SQLite database within the Colab runtime
conn = sqlite3.connect('openbind_assay.db')

# Load the cleaned data into db
df_clean.to_sql('assay_metrics', conn, if_exists='replace', index=False)

# Execute SQL queries to display the compounds with the highest pKD binding affinities
query = """
SELECT complex_name, smiles, experimental_pKD
FROM assay_metrics
ORDER BY experimental_pKD DESC
LIMIT 5;
"""

print("Executing SQL Query for Top Affinity Compounds (Highest experimental_pKD)...")
top_compounds = pd.read_sql(query, conn)
display(top_compounds)

# Close the connection
conn.close()